# Notebook 2 — More than one unknown: straight-line fitting

**What you will practise here:**

- a model with three unknowns instead of one
- `pm.Deterministic` — naming a quantity that is calculated, not sampled
- **pair plots** — seeing two parameters trade off against each other
- fixing a trade-off by rewriting the model (this is called **reparameterising**)
- `pm.Data` + `pm.set_data` — predicting at new inputs

**The example:** points scattered around a straight line, `y = a + b*x + noise`.
Three unknowns: the intercept `a`, the slope `b`, and the noise size `sigma`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az

AZ_MAJOR = int(az.__version__.split(".")[0])
plot_ppc = getattr(az, "plot_ppc", None) or az.plot_ppc_dist

print("pymc", pm.__version__, "| arviz", az.__version__)

## Step 1 — Fake data, known answer

Note that `x` runs from 20 to 30, not from 0 to 10. That choice is deliberate and it
will cause a problem later. Watch for it.

In [ ]:
rng = np.random.default_rng(0)

TRUE_A, TRUE_B, TRUE_SIGMA = 5.0, 1.3, 2.0
N = 60

x = rng.uniform(20, 30, N)
y = TRUE_A + TRUE_B * x + rng.normal(0, TRUE_SIGMA, N)

plt.figure(figsize=(6, 3.5))
plt.scatter(x, y, s=18)
plt.xlabel("x"); plt.ylabel("y"); plt.title("The fake data")
plt.show()

## Step 2 — The model

`pm.Data("x_data", x)` puts `x` into a named slot inside the model. Later we can swap in
different `x` values without rebuilding the model. That is how you predict at new inputs.

`pm.Deterministic("mu", ...)` says: *"mu is not an unknown, it is calculated from a and b —
but please save it in the output so I can look at it."* Handy, but it does make the output
file bigger, so only do it for things you actually want.

`pm.HalfNormal` is a Normal distribution cut off at zero. Use it for anything that cannot
be negative, like a standard deviation.

In [ ]:
with pm.Model() as line_model:
    x_data = pm.Data("x_data", x)

    a = pm.Normal("a", mu=0, sigma=20)        # intercept
    b = pm.Normal("b", mu=0, sigma=5)         # slope
    sigma = pm.HalfNormal("sigma", sigma=5)   # noise size

    mu = pm.Deterministic("mu", a + b * x_data)

    pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y, shape=x_data.shape[0])

try:
    display(pm.model_to_graphviz(line_model))
except Exception as e:
    print("no graphviz:", e)

## Step 3 — Prior predictive check

Before sampling, ask: what lines does my prior allow? Draw 60 of them.
If they cover the data comfortably, the prior is fine. If they all miss by miles,
the prior is fighting the data and you should widen it or move it.

In [ ]:
with line_model:
    prior = pm.sample_prior_predictive(200, random_seed=1)

pa = np.asarray(prior.prior["a"]).ravel()
pb = np.asarray(prior.prior["b"]).ravel()

xg = np.linspace(20, 30, 20)
plt.figure(figsize=(6, 3.5))
for i in range(60):
    plt.plot(xg, pa[i] + pb[i] * xg, color="grey", alpha=.25, lw=.8)
plt.scatter(x, y, s=18, color="C1", zorder=5, label="data")
plt.xlabel("x"); plt.ylabel("y")
plt.title("Lines the prior allows")
plt.legend(); plt.show()

## Step 4 — Sample and check

In [ ]:
with line_model:
    idata = pm.sample(2000, tune=1000, chains=4, cores=1, random_seed=2)

# var_names keeps 'mu' (60 values!) out of the table
az.summary(idata, var_names=["a", "b", "sigma"])

In [ ]:
print(f"true a = {TRUE_A},  true b = {TRUE_B},  true sigma = {TRUE_SIGMA}")
print()
print("divergences:", int(idata.sample_stats["diverging"].sum()))

Look at `ess_bulk` for `a` and `b`. It is probably much lower than for `sigma`,
even though all three came from the same run. That is the clue that something is
awkward about `a` and `b`. The next plot shows what.

In [ ]:
az.plot_trace(idata, var_names=["a", "b", "sigma"])
plt.tight_layout(); plt.show()

## Step 5 — The pair plot: two parameters that trade off

A **pair plot** puts one parameter on each axis and shows every draw as a dot.
If the cloud is a round blob, the two parameters are independent — good.
If it is a thin diagonal streak, they are **correlated**: you can raise one and lower the
other and the model fits the data just as well. The sampler has to crawl along that
narrow streak, which is slow, and that is exactly why `ess_bulk` was low.

In [ ]:
a_draws = np.asarray(idata.posterior["a"]).ravel()
b_draws = np.asarray(idata.posterior["b"]).ravel()

plt.figure(figsize=(4.5, 4))
plt.scatter(a_draws, b_draws, s=3, alpha=.2)
plt.xlabel("a (intercept)"); plt.ylabel("b (slope)")
plt.title(f"correlation = {np.corrcoef(a_draws, b_draws)[0,1]:.3f}")
plt.show()

# ArviZ has a built-in version too
az.plot_pair(idata, var_names=["a", "b"])
plt.show()

### Why does this happen?

`a` is the height of the line **at x = 0**. But our data lives between x = 20 and x = 30.
So x = 0 is far off to the left, and the line is being extended a long way to get there.
Tilt the slope slightly and the intercept has to move a lot to compensate.

The model is not wrong. It is just written in an awkward way.

## Step 6 — Fix it by reparameterising

**Reparameterising** means: write the same model using different unknowns, chosen so the
sampler has an easier time.

Here the fix is one line. Instead of `a + b*x`, use `a_c + b*(x - x_mean)`.
Now `a_c` is the height of the line **in the middle of the data**, where we actually have
information. The trade-off disappears.

Same model, same predictions, much better sampling. This is worth remembering — when a
sampler struggles, the answer is often to rewrite the model, not to run it longer.

In [ ]:
x_mean = x.mean()

with pm.Model() as line_model_c:
    xc = pm.Data("xc", x - x_mean)

    a_c = pm.Normal("a_c", mu=0, sigma=50)     # height at the middle of the data
    b = pm.Normal("b", mu=0, sigma=5)
    sigma = pm.HalfNormal("sigma", sigma=5)

    mu = a_c + b * xc
    pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y, shape=xc.shape[0])

    idata_c = pm.sample(2000, tune=1000, chains=4, cores=1, random_seed=3)

az.summary(idata_c, var_names=["a_c", "b", "sigma"])

In [ ]:
ac_draws = np.asarray(idata_c.posterior["a_c"]).ravel()
bc_draws = np.asarray(idata_c.posterior["b"]).ravel()

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].scatter(a_draws, b_draws, s=3, alpha=.2)
ax[0].set_title(f"before: r = {np.corrcoef(a_draws, b_draws)[0,1]:.3f}")
ax[0].set_xlabel("a"); ax[0].set_ylabel("b")
ax[1].scatter(ac_draws, bc_draws, s=3, alpha=.2, color="C2")
ax[1].set_title(f"after centring: r = {np.corrcoef(ac_draws, bc_draws)[0,1]:.3f}")
ax[1].set_xlabel("a_c"); ax[1].set_ylabel("b")
plt.tight_layout(); plt.show()

print("ESS for b, before:", float(az.ess(idata,   var_names=["b"])["b"]))
print("ESS for b, after :", float(az.ess(idata_c, var_names=["b"])["b"]))

### Recovering the original intercept

You did not lose `a`. It is just `a_c - b * x_mean`. Compute it **per draw**, so the
uncertainty comes along with it.

In [ ]:
a_recovered = ac_draws - bc_draws * x_mean

print(f"true a                  : {TRUE_A}")
print(f"a from first model      : {a_draws.mean():.3f}")
print(f"a rebuilt from centred  : {a_recovered.mean():.3f}")

## Step 7 — Predicting at new x values

`pm.set_data` swaps new numbers into the named slot, then
`pm.sample_posterior_predictive` runs every posterior draw through the model.

You get two different bands, and mixing them up is a common mistake:

- **`mu` band** — uncertainty about *where the line is*. Narrow.
- **`y_obs` band** — uncertainty about *where the next data point will land*.
  Wider, because it also includes the noise `sigma`.

In [ ]:
x_new = np.linspace(18, 32, 40)

with line_model_c:
    pm.set_data({"xc": x_new - x_mean})
    pred = pm.sample_posterior_predictive(
        idata_c, var_names=["y_obs"], random_seed=4, predictions=True
    )

y_pred = np.asarray(pred.predictions["y_obs"]).reshape(-1, len(x_new))

# line uncertainty, computed by hand from the draws
mu_pred = ac_draws[:, None] + bc_draws[:, None] * (x_new - x_mean)[None, :]

plt.figure(figsize=(7, 4))
plt.fill_between(x_new, *np.percentile(y_pred, [2.5, 97.5], axis=0),
                 alpha=.2, color="C0", label="95% for a new data point")
plt.fill_between(x_new, *np.percentile(mu_pred, [2.5, 97.5], axis=0),
                 alpha=.5, color="C0", label="95% for the line itself")
plt.plot(x_new, mu_pred.mean(axis=0), color="C0", lw=2, label="posterior mean line")
plt.scatter(x, y, s=18, color="C1", zorder=5, label="data")
plt.xlabel("x"); plt.ylabel("y"); plt.legend()
plt.title("Two kinds of uncertainty")
plt.show()

## Step 8 — Posterior predictive check

In [ ]:
with line_model_c:
    pm.set_data({"xc": x - x_mean})   # put the original x back first!
    pm.sample_posterior_predictive(idata_c, random_seed=5, extend_inferencedata=True)

plot_ppc(idata_c)
plt.show()

## Your turn — exercises

1. **Shift the data further out.** Change `x = rng.uniform(20, 30, N)` to
   `rng.uniform(200, 210, N)` and re-run the *uncentred* model. How bad does the
   correlation and the ESS get?
2. **Cut the data down.** Set `N = 8`. How wide does the posterior for `b` become?
   Does the true slope still sit inside the interval?
3. **Add a fake third parameter** that the data cannot see, e.g. `c = pm.Normal("c", 0, 1)`
   that appears nowhere in `mu`. Sample and look at its posterior. It will be identical to
   its prior. That is the signature of a parameter the data says nothing about.
4. **Set a stupid prior:** `b = pm.Normal("b", mu=-5, sigma=0.1)`. The prior insists the
   slope is negative; the data says it is positive. Who wins? Look at the posterior
   predictive check — the model will visibly fail to fit.
5. Swap the Normal noise for `pm.StudentT("y_obs", nu=3, mu=mu, sigma=sigma, observed=y)`,
   then add one wild outlier to `y`. Compare how much the slope moves under each.